# Day 7 Lab 08: CDC Latest Order State

        **Layer:** Silver  
        **Python reference:** `Lab_Files/labs/lab_08_cdc_latest_order_state.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Read valid Silver order events.
- Explicitly deduplicate duplicate source events before CDC.
- Use a window function helper to select the latest state per order.
- Write and preview the current-state Silver table.

**Dependency:** Run Lab/Notebook 07 first so `lake/silver/orders_valid` exists.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [ ]:
from pathlib import Path
import os
import sys

HERE = Path.cwd().resolve()
LAB_FILES = HERE / "Lab_Files"
if not LAB_FILES.exists():
    LAB_FILES = HERE.parent / "Lab_Files"

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")

## 1. Import Lab Helpers

In [ ]:
from day7_common import LAKE_DIR, OUTPUT_DIR, deduplicate_order_events, ensure_output_dirs, latest_order_state, require_source_data, spark_session, write_csv_dir

## 2. Start Spark and Read Valid Orders

In [ ]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook08CdcLatestState")

valid = spark.read.parquet(str(LAKE_DIR / "silver" / "orders_valid"))
valid.select("order_id", "event_id", "status", "event_time_ts").orderBy("order_id", "event_time_ts").show(30, truncate=False)

## 3. Deduplicate Valid Events

In [ ]:
deduplicated = deduplicate_order_events(valid)
dedup_preview = deduplicated.select("order_id", "event_id", "status", "amount", "currency", "event_time_ts").orderBy("order_id", "event_time_ts", "event_id")
dedup_preview.show(30, truncate=False)
print(f"Valid rows before deduplication: {valid.count()}")
print(f"Rows after explicit Silver deduplication: {deduplicated.count()}")

## 4. Calculate and Inspect Current State

In [ ]:
current = latest_order_state(deduplicated)
preview = current.select("order_id", "event_id", "status", "amount", "currency", "event_time_ts").orderBy("order_id")
preview.show(20, truncate=False)

## 5. Write Current-State Silver Table

In [ ]:
deduplicated.write.mode("overwrite").parquet(str(LAKE_DIR / "silver" / "orders_deduplicated"))
current_path = LAKE_DIR / "silver" / "orders_current"
current.write.mode("overwrite").parquet(str(current_path))
write_csv_dir(dedup_preview, OUTPUT_DIR / "notebook_08_deduplicated_events_preview")
write_csv_dir(preview, OUTPUT_DIR / "notebook_08_orders_current_preview")
print(f"Current order rows: {current.count()}")

## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [ ]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()